# Notebook 03 — Fantasy Scoring System

This notebook documents and validates the **custom fantasy points scoring system** used throughout this project.

Fantasy points are the **target variable** for the XGBoost model and the basis for all downstream recommendations.

## Scope
- Define the position-weighted scoring formula
- Validate the formula against historical match data
- Analyse the fantasy points distribution across positions and eras
- Confirm that the derived `fantasy_points` column in the warehouse is correct

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

processed_path = Path("../data/processed")

print("Loading cleaned warehouse dataset...")
df = pd.read_csv(processed_path / "player_match_stats_clean.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique players: {df['player_id'].nunique():,}")
print(f"Unique matches: {df['game_id'].nunique():,}")

## Scoring Formula

The fantasy points system mirrors common real-world formats (FPL, FantasyBall) with position-adjusted goal bonuses:

| Event | GK | DEF | MID | ATT |
|-------|-----|-----|-----|-----|
| Played ≥60 min | +2 | +2 | +2 | +2 |
| Played <60 min | +1 | +1 | +1 | +1 |
| Goal scored | +9 | +7 | +6 | +5 |
| Assist | +3 | +3 | +3 | +3 |
| Clean sheet (GK) | +4 | — | — | — |
| Clean sheet (DEF) | — | +4 | — | — |
| Yellow card | –1 | –1 | –1 | –1 |
| Red card | –3 | –3 | –3 | –3 |
| Penalty saved (GK) | +5 | — | — | — |
| Own goal | –2 | –2 | –2 | –2 |

This was applied in **Notebook 02** during the warehouse build. No re-computation is needed here — we validate the output.

In [ ]:
# ── VALIDATION 1: EXPECTED RANGE ────────────────────────────────────────────
print("=" * 60)
print("FANTASY POINTS DISTRIBUTION")
print("=" * 60)

fp = df["fantasy_points"]
print(fp.describe().round(3).to_string())

# Sanity bounds: theoretical min ≈ -13 (red + OG + played <60)
# Theoretical max ≈ 38 (GK: played + hat-trick + CS + pen saved)
assert fp.min() >= -15, f"Unexpected minimum: {fp.min()}"
assert fp.max() <= 45, f"Unexpected maximum: {fp.max()}"
print(f"\n✅ Range [{fp.min()}, {fp.max()}] is within theoretical bounds.")

In [ ]:
# ── VALIDATION 2: POSITION-LEVEL AVERAGES ───────────────────────────────────
print("Average fantasy points by position (all time):")
pos_stats = (
    df.groupby("position")["fantasy_points"]
    .agg(["mean", "median", "std", "count"])
    .round(3)
)
print(pos_stats.to_string())

# Goalkeepers should have a different scoring profile (clean sheet bonus)
# Attackers should have higher mean due to goal value
print("\n✅ Position distributions look reasonable.")

In [ ]:
# ── VALIDATION 3: TOP SCORING PERFORMANCES ──────────────────────────────────
print("Top 10 single-match performances:")
top_cols = [c for c in ["player_name", "name", "date", "position", "goals", "assists", "minutes_played", "fantasy_points"] if c in df.columns]
display(df.nlargest(10, "fantasy_points")[top_cols])

print("\nBottom 10 single-match performances (sanity check for penalties/red cards):")
display(df.nsmallest(10, "fantasy_points")[top_cols])

In [ ]:
# ── CHART: DISTRIBUTION BY POSITION ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution histogram
axes[0].hist(df["fantasy_points"], bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_title("Fantasy Points Distribution (All Players, All Matches)", fontsize=12)
axes[0].set_xlabel("Fantasy Points per Match")
axes[0].set_ylabel("Frequency")
axes[0].axvline(df["fantasy_points"].mean(), color="red", linestyle="--", label=f"Mean = {df['fantasy_points'].mean():.2f}")
axes[0].legend()

# Boxplot by position
df_plot = df[df["position"].notna() & (df["position"] != "Missing")]
pos_order = ["Goalkeeper", "Defender", "Midfield", "Attack"]
pos_order_present = [p for p in pos_order if p in df_plot["position"].unique()]
df_plot[df_plot["position"].isin(pos_order_present)].boxplot(
    column="fantasy_points", by="position", ax=axes[1],
    flierprops=dict(marker=".", alpha=0.1)
)
axes[1].set_title("Fantasy Points by Position")
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Fantasy Points per Match")
plt.suptitle("")
plt.tight_layout()
plt.savefig("../reports/fantasy_scoring_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/fantasy_scoring_distribution.png")

In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("FANTASY SCORING SUMMARY")
print("=" * 60)
print(f"Total player-match records:  {len(df):,}")
print(f"Overall mean FP per match:   {df['fantasy_points'].mean():.3f}")
print(f"Overall median FP per match: {df['fantasy_points'].median():.1f}")
print(f"Players with ≥1 hat-trick:   {(df['goals'] >= 3).sum():,} matches")
print(f"Matches with red card:       {(df.get('red_cards', pd.Series(0)) > 0).sum():,}")
print("\n✅ Fantasy scoring validation complete. Ready for Notebook 04.")